# 07 · Cross-validation & ablation (PRD §13.1–13.2)
**Runs:** locally (~10–30 min). **Produces:** `reports/cv_comparison.json`, `reports/ablation.csv`.

15 product-grouped folds (5 folds × 3 repeats). Baseline = Ridge on the star rating alone.
Components are kept only if they improve MAE in ≥ 60% of folds; negative results stay in the table.

In [ ]:
# --- Repo setup: make `app` (backend) and `vera_training` importable ---------
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "PRD.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ("backend", "training"):
    if str(ROOT / sub) not in sys.path:
        sys.path.insert(0, str(ROOT / sub))

import os
if not (ROOT / ".env").exists():  # app settings require these; real values come from .env
    os.environ.setdefault("DATABASE_URL", "sqlite:///:memory:")
    os.environ.setdefault("REDIS_URL", "redis://localhost:6379/0")

from vera_training import paths
paths.ensure_dirs()
print("Repo root:", ROOT)

In [ ]:
import json
import joblib
from sklearn.linear_model import Ridge
from vera_training import evaluation
from vera_training.fusion import BootstrapEnsemble

data = joblib.load(paths.FUSION_CACHE)["matrices"]["all"]
X, y, groups = data["X"], data["y"], data["frame"].parent_asin.tolist()

baseline = evaluation.cross_validated_mae(X[:, :1], y, groups, Ridge)
vera = evaluation.cross_validated_mae(X, y, groups, lambda: BootstrapEnsemble(n_members=20))
comparison = evaluation.paired_comparison(baseline, vera)
(paths.REPORTS / "cv_comparison.json").write_text(json.dumps(comparison, indent=2))
print(json.dumps(comparison, indent=2))

In [ ]:
table = evaluation.ablation_study(X, y, groups, lambda: BootstrapEnsemble(n_members=5))
table.to_csv(paths.REPORTS / "ablation.csv", index=False)
table